# ĐỒ ÁN 1: MA TRẬN VÀ TÍNH TOÁN KHOA HỌC
---
## Demo Phần 1: Phép khử Gauss và Các Ứng Dụng
**Môn học:** Toán Ứng Dụng và Thống Kê
**Thực hiện bởi:** Nhóm 15

In [41]:
import numpy as np
EPSILON = 1e-9

---
### 1.1 Thuật toán RREF
**Cài đặt Hàm `gaussian_eliminate(A, b)`:**
Đưa ma trận mở rộng $[A|b]$ về dạng Bậc thang dòng rút gọn (RREF) sử dụng phép khử Gauss-Jordan kết hợp chọn phần tử chốt (Partial Pivoting) để giảm thiểu sai số.

Hàm tự động phát hiện ma trận suy biến và trả về các thông số bổ trợ (`swap_count`, `det_multiplier`) cho các ứng dụng ở phần sau.

In [42]:
def gaussian_eliminate(A, b):
    """
    Đưa ma trận [A|b] về RREF bằng Gauss-Jordan (có Partial Pivoting).

    Args:
        A (list of lists): Ma trận hệ số.
        b (list/list of lists): Vector hệ số tự do hoặc ma trận ghép (VD: ma trận I).

    Returns:
        Trả về tuple (RREF_matrix, x, swap_count, det_multiplier):
        - RREF_matrix: Ma trận RREF.
        - x: Nghiệm hệ phương trình (list) hoặc chuỗi báo lỗi ("Vo nghiem"...).
        - swap_count: Số lần hoán đổi dòng.
        - det_multiplier: Tích các pivot (dùng để tính định thức).
    """
    m = len(A)
    n = len(A[0])
    epsilon = 1e-9

    # 1. Tạo ma trận mở rộng M = [A | b]
    M = []

    # Đếm số cột thực sự của b để biết là vector hay ma trận
    if isinstance(b[0], list):
        b_cols = len(b[0])
    else:
        b_cols = 1

    is_b_matrix = (b_cols > 1)

    for r in range(m):
        row = [float(val) for val in A[r]]
        # Nối thêm phần hệ số tự do b
        if isinstance(b[r], list):
            row.extend([float(val) for val in b[r]])
        else:
            row.append(float(b[r]))
        M.append(row)

    swap_count = 0
    det_multiplier = 1.0
    i = 0

    # 2. Bắt đầu khử
    for k in range(n):
        if i >= m:
            break

        # Tìm phần tử có giá trị tuyệt đối lớn nhất trong cột
        max_row = i
        max_val = abs(M[i][k])
        for r in range(i + 1, m):
            if abs(M[r][k]) > max_val:
                max_val = abs(M[r][k])
                max_row = r

        # Xử lý nếu cột này toàn số 0
        if max_val < epsilon:
            print(f"Khong co pivot tai cot {k}")
            det_multiplier = 0.0
            continue

        # Check nếu cần hoán đổi dòng
        if max_row != i:
            M[i], M[max_row] = M[max_row], M[i]
            swap_count += 1

        # Sử dụng lại pivot để tính định thức
        pivot = M[i][k]
        det_multiplier *= pivot

        # Chuẩn hóa pivot
        for c in range(k, len(M[0])):
            M[i][c] /= pivot

        # Khử các phần tử nằm trên và dưới pivot về 0
        for r in range(m):
            if r != i:
                factor = M[r][k]
                for c in range(k, len(M[0])):
                    M[r][c] -= factor * M[i][c]
        i += 1

    RREF_matrix = M

    # 3. Phân tích nghiệm cơ bản
    x = "Khong xac dinh"
    if not is_b_matrix:
        is_inconsistent = False
        for r in range(m):
            # Kiểm tra xem có dòng nào dạng [0 0 ... 0 | k] với k != 0 không
            all_zero_left = all(abs(M[r][c]) < epsilon for c in range(n))
            if all_zero_left and abs(M[r][n]) >= epsilon:
                is_inconsistent = True
                break

        if is_inconsistent:
            x = "Vo nghiem"
        elif i < n:  # Số pivot (i) < Số biến (n) -> Vô số nghiệm
            x = "Vo so nghiem"
        else:
            x = [M[r][n] for r in range(n)]

    return (RREF_matrix, x, swap_count, det_multiplier)

---
### 1.2 Giải Hệ Phương Trình và Nghiệm Tổng Quát
**Cài đặt Hàm `back_substitution(U, c)` và `solve_system(RREF_matrix)`:**
Phân loại nghiệm (Duy nhất, Vô nghiệm, Vô số nghiệm) dựa trên hình dáng của ma trận RREF. Đặc biệt, thuật toán có khả năng trích xuất các biến tự do và xuất ra chuỗi công thức nghiệm tổng quát bằng các tham số $t_i$.

In [43]:
def back_substitution(U, c):
    """
    Thực hiện phép thế ngược trên ma trận tam giác trên U để giải hệ Ux = c.

    Args:
        U (list of lists): Ma trận tam giác trên.
        c (list): Vector hệ số tự do.

    Returns:
        list: Nghiệm của hệ phương trình.

    Raises:
        ValueError: Nếu ma trận U không khả nghịch (pivot bằng 0).
    """
    m = len(c)
    solution = [0.0] * m
    for i in range(m - 1, -1, -1):
        if abs(U[i][i]) < EPSILON:
            raise ValueError("He phuong trinh khong co nghiem duy nhat!")
        total = 0
        for j in range(i + 1, m):
            total += U[i][j] * solution[j]
        solution[i] = (c[i] - total) / U[i][i]
    return solution

In [44]:
def solve_system(RREF_matrix):
    """
    Phân tích ma trận RREF để xác định nghiệm của hệ phương trình tuyến tính.

    Args:
        RREF_matrix (list of lists): Ma trận đã được đưa về dạng RREF (Reduced Row Echelon Form).

    Returns:
        str: Chuỗi mô tả nghiệm hệ phương trình. Có thể là:
             - "Ma tran rong" nếu ma trận rỗng.
             - "Phuong trinh vo nghiem" nếu hệ vô nghiệm.
             - "He co nghiem duy nhat: (x_1, x_2, ...) = (val1, val2, ...)" nếu có nghiệm duy nhất.
             - Chuỗi biểu diễn nghiệm tổng quát với biến tự do nếu có vô số nghiệm.
    """
    if not RREF_matrix or not RREF_matrix[0]:
        return "Ma tran rong"

    res_str = ""
    rows = len(RREF_matrix)
    cols = len(RREF_matrix[0])
    n = cols - 1  # Số lượng biến

    # Kiểm tra vô nghiệm
    for i in range(rows):
        isAllZero = all(abs(RREF_matrix[i][j]) < EPSILON for j in range(n))
        if (isAllZero and abs(RREF_matrix[i][n]) > EPSILON):
            return "Phuong trinh vo nghiem"

    # Tìm các cột chứa pivot
    pivots = {}
    for i in range(rows):
        for j in range(n):
            if abs(RREF_matrix[i][j]) > EPSILON:
                pivots[i] = j
                break

    # Trường hợp 1: Nghiệm duy nhất
    if len(pivots) == n:
        solution = [0.0] * n
        for r, c in pivots.items():
            solution[c] = RREF_matrix[r][n]
        var_names = ", ".join([f"x_{i + 1}" for i in range(n)])
        sol_values = ", ".join([f"{val:.4g}" for val in solution])
        return f"He co nghiem duy nhat: ({var_names}) = ({sol_values})"

    # Trường hợp 2: Vô số nghiệm
    free_vars = [j for j in range(n) if j not in pivots.values()]
    free_var_symbols = {val: f"t_{idx + 1}" for idx, val in enumerate(free_vars)}

    for j in range(n):
        if j not in pivots.values():
            res_str += f"x_{j + 1} = {free_var_symbols[j]}\n"
        else:
            currentRow = [r for r, c in pivots.items() if c == j][0]
            constant = RREF_matrix[currentRow][n]
            eq_parts = []

            if abs(constant) > EPSILON:
                eq_parts.append(f"{constant:.4g}")

            for free_var in free_vars:
                coef = -RREF_matrix[currentRow][free_var]
                if abs(coef) > EPSILON:
                    sign = "+ " if coef > 0 else "- "
                    coef = abs(coef)
                    part = f"{sign}{free_var_symbols[free_var]}" if abs(
                        coef - 1) < EPSILON else f"{sign}{coef:.4g}{free_var_symbols[free_var]}"
                    if not eq_parts and sign == "+ ":
                        part = part.strip("+ ")
                    eq_parts.append(part.strip())

            if not eq_parts:
                eq_parts.append("0")
            res_str += f"x_{j + 1} = {' '.join(eq_parts)}\n"

    return res_str

---
### 1.3 Các Ứng dụng Đại số
**Các hàm ứng dụng:**
* `determinant(A)`: Tính định thức thông qua phép khử Gauss.
* `inverse(A)`: Tìm ma trận nghịch đảo bằng cách ghép $[A|I]$.
* `rank_and_basis(A)`: Trích xuất Hạng và Cơ sở (Không gian dòng, Không gian cột, Không gian nghiệm).


In [45]:
def determinant(A):
    """
    [TÍNH ĐỊNH THỨC] Gọi hàm Gauss để lấy tích các pivot và số lần hoán đổi.
    Công thức: det(A) = (-1)^swap_count * det_multiplier

    Args:
        A (list of lists): Ma trận vuông cần tính định thức.

    Returns:
        float or str: Giá trị định thức (float) nếu ma trận vuông và khả nghịch, hoặc chuỗi "Ma tran khong vuong, khong tinh duoc dinh thuc" nếu ma trận không vuông.
    """
    m = len(A)
    n = len(A[0])
    if m != n:
        return "Ma tran khong vuong, khong tinh duoc dinh thuc"

    # Truyền b là một vector cột toàn 0
    b_dummy = [0.0] * m
    _, _, swap_count, det_multiplier = gaussian_eliminate(A, b_dummy)

    det = ((-1) ** swap_count) * det_multiplier

    # Tránh in ra số -0.0
    return 0.0 if abs(det) < 1e-9 else det

In [46]:
def inverse(A):
    """
    [TÌM MA TRẬN NGHỊCH ĐẢO] Giải ma trận [A|I] bằng Gauss-Jordan.
    Lấy nửa bên phải của RREF làm ma trận nghịch đảo.

    Args:
        A (list of lists): Ma trận vuông cần tìm nghịch đảo.

    Returns:
        list of lists or str: Ma trận nghịch đảo (list of lists) nếu ma trận khả nghịch, 
        hoặc chuỗi lỗi ("Ma tran khong vuong" hoặc "Ma tran suy bien, khong co nghich dao") nếu không thể tìm nghịch đảo.
    """
    m = len(A)
    n = len(A[0])
    if m != n:
        return "Ma tran khong vuong"

    # 1. Tạo ma trận đơn vị I kích thước n x n
    I = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

    # 2. Gọi Gauss đưa [A | I] về RREF
    RREF_matrix, _, _, det_multiplier = gaussian_eliminate(A, I)

    # 3. Nếu định thức = 0 thì không có nghịch đảo
    if abs(det_multiplier) < 1e-9:
        return "Ma tran suy bien, khong co nghich dao"

    # 4. Trích xuất nửa bên phải của ma trận RREF (A^-1)
    inv_matrix = []
    for r in range(m):
        inv_matrix.append(RREF_matrix[r][n:])

    return inv_matrix

In [47]:
def rank_and_basis(A):
    """
    [HẠNG VÀ CƠ SỞ] Tính hạng và trích xuất cơ sở Không gian Cột, Dòng, Nghiệm.

    Args:
        A (list of lists): Ma trận cần phân tích.

    Returns:
        dict: Từ điển chứa các khóa sau:
            - 'rank' (int): Hạng của ma trận.
            - 'column_space_basis' (list of lists): Cơ sở của không gian cột (danh sách các vector cột pivot).
            - 'row_space_basis' (list of lists): Cơ sở của không gian dòng (danh sách các hàng không zero trong RREF).
            - 'null_space_basis' (list of lists): Cơ sở của không gian nghiệm (danh sách các vector nghiệm cơ bản).
    """
    m = len(A)
    n = len(A[0])
    epsilon = 1e-9

    b_dummy = [0.0] * m
    RREF_matrix, _, _, _ = gaussian_eliminate(A, b_dummy)

    # 1. Tìm các cột chứa pivot
    pivot_cols = []
    r_idx = 0
    for c_idx in range(n):
        if r_idx >= m:
            break
        if abs(RREF_matrix[r_idx][c_idx]) > epsilon:
            pivot_cols.append(c_idx)
            r_idx += 1

    rank = len(pivot_cols)

    # 2. Cơ sở không gian Cột
    column_space = [[A[r][c] for r in range(m)] for c in pivot_cols]

    # 3. Cơ sở không gian Dòng
    row_space = []
    for r in range(m):
        if any(abs(val) > epsilon for val in RREF_matrix[r][:n]):
            row_space.append(RREF_matrix[r][:n])

    # 4. Cơ sở không gian Nghiệm
    free_cols = [c for c in range(n) if c not in pivot_cols]
    null_space = []
    for fc in free_cols:
        vec = [0.0] * n
        vec[fc] = 1.0
        for idx, pc in enumerate(pivot_cols):
            vec[pc] = -RREF_matrix[idx][fc]
        null_space.append(vec)

    return {
        'rank': rank,
        'column_space_basis': column_space,
        'row_space_basis': row_space,
        'null_space_basis': null_space
    }

---
## Phần 2: Kiểm tra kết quả
**Nhiệm vụ:** Sử dụng thư viện `NumPy` để kiểm tra kết quả. Mỗi hàm bên trên sẽ có 5 bộ test cases khác nhau.

Kết quả trả về **[ĐẠT]** cho thấy thuật toán của nhóm chạy chính xác và khớp với sai số tiêu chuẩn của thư viện Python.

### 2.1. Kiểm thử hàm Giải hệ phương trình
Kiểm tra khả năng khử Gauss và biện luận nghiệm của hệ $Ax = b$.

In [48]:
def verify_solve(A, b, x_nhom):
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float).flatten()
    n = A_np.shape[1]

    rank_A = np.linalg.matrix_rank(A_np)
    rank_Ab = np.linalg.matrix_rank(np.column_stack((A_np, b_np)))

    if rank_A < rank_Ab:
        expected = "Vo nghiem"
    elif rank_A == rank_Ab and rank_A < n:
        expected = "Vo so nghiem"
    else:
        expected = "Nghiem duy nhat"

    if isinstance(x_nhom, str) and "Vo nghiem" in x_nhom:
        return "[ĐẠT]" if expected == "Vo nghiem" else f"[CHƯA ĐẠT] (NumPy ra {expected})"
    elif isinstance(x_nhom, str) and "Vo so nghiem" in x_nhom:
        return "[ĐẠT]" if expected == "Vo so nghiem" else f"[CHƯA ĐẠT] (NumPy ra {expected})"
    elif isinstance(x_nhom, list):
        if expected == "Nghiem duy nhat":
            x_np = np.linalg.solve(A_np, b_np)
            return "[ĐẠT]" if np.allclose(x_nhom, x_np, atol=1e-9) else "[CHƯA ĐẠT] (Sai số nghiệm lớn)"
        return f"[CHƯA ĐẠT] (Nhóm ra nghiệm duy nhất, NumPy ra {expected})"
    return "[CHƯA ĐẠT]"


def test_gaussian_eliminate():
    print("\n" + "="*60)
    print("--- TEST HÀM GIẢI HỆ PHƯƠNG TRÌNH (gaussian_eliminate) ---")
    print("="*60)

    # TC1: Bình thường 3x3
    A1, b1 = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], [[8], [-11], [-3]]
    _, x1, _, _ = gaussian_eliminate(A1, b1)
    print("TC1 (Nghiệm duy nhất):")
    print(f"  + Kết quả : {x1}")
    print(f"  + Đánh giá     : {verify_solve(A1, b1, x1)}\n")

    # TC2: Vô số nghiệm
    A2, b2 = [[1, 2, 3], [4, 5, 6], [5, 7, 9]], [[1], [2], [3]]
    _, x2, _, _ = gaussian_eliminate(A2, b2)
    print("TC2 (Vô số nghiệm):")
    print(f"  + Kết quả : {x2}")
    print(f"  + Đánh giá     : {verify_solve(A2, b2, x2)}\n")

    # TC3: Vô nghiệm
    A3, b3 = [[1, 2, 3], [4, 5, 6], [5, 7, 9]], [[1], [2], [100]]
    _, x3, _, _ = gaussian_eliminate(A3, b3)
    print("TC3 (Vô nghiệm):")
    print(f"  + Kết quả : {x3}")
    print(f"  + Đánh giá     : {verify_solve(A3, b3, x3)}\n")

    # TC4: Ma trận chữ nhật (4 phương trình, 3 ẩn - có nghiệm)
    A4, b4 = [[1, 2, 3], [2, 5, 2], [1, 1, 7], [3, 7, 5]], [[14], [18], [22], [32]]
    _, x4, _, _ = gaussian_eliminate(A4, b4)
    print("TC4 (Ma trận chữ nhật 4x3):")
    print(f"  + Kết quả : {x4}")
    print(f"  + Đánh giá     : {verify_solve(A4, b4, x4)}\n")

    # TC5: Ngoại lệ Pivot (Cột toàn 0)
    A5, b5 = [[0, 2, 3], [0, 5, 6], [0, 8, 9]], [[5], [11], [17]]
    _, x5, _, _ = gaussian_eliminate(A5, b5)
    print("TC5 (Suy biến cột đầu):")
    print(f"  + Kết quả : {x5}")
    print(f"  + Đánh giá     : {verify_solve(A5, b5, x5)}\n")

### 2.2. Kiểm thử hàm Tính Định thức
Kiểm tra khả năng theo dõi số lần hoán đổi dòng và tính định thức cho ma trận vuông.

In [49]:
def verify_det(A, det_nhom):
    if len(A) != len(A[0]):
        return "[ĐẠT]" if isinstance(det_nhom, str) else "[CHƯA ĐẠT] (Không báo lỗi ma trận không vuông)"
    det_np = np.linalg.det(np.array(A, dtype=float))
    return "[ĐẠT]" if np.isclose(det_nhom, det_np, atol=1e-5) else f"[CHƯA ĐẠT] (NumPy ra: {det_np})"


def test_determinant():
    print("\n" + "="*60)
    print("--- TEST HÀM TÍNH ĐỊNH THỨC (determinant) ---")
    print("="*60)

    A1 = [[1, 2], [3, 4]]
    det1 = determinant(A1)
    print(f"TC1 (Ma trận 2x2)           | Det: {det1:<6} | {verify_det(A1, det1)}")

    A2 = [[2, 0, 0], [0, 3, 0], [0, 0, 4]]
    det2 = determinant(A2)
    print(f"TC2 (Ma trận đường chéo)    | Det: {det2:<6} | {verify_det(A2, det2)}")

    A3 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
    det3 = determinant(A3)
    print(f"TC3 (Suy biến det=0)        | Det: {det3:<6} | {verify_det(A3, det3)}")

    A4 = [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]
    det4 = determinant(A4)
    print(f"TC4 (Ma trận đơn vị 4x4)    | Det: {det4:<6} | {verify_det(A4, det4)}")

    A5 = [[1, 2, 3], [4, 5, 6]]
    det5 = determinant(A5)
    print(f"TC5 (Ma trận không vuông)   | Det: {det5:<6} | {verify_det(A5, det5)}")

### 2.3. Kiểm thử hàm Tìm Ma trận Nghịch đảo
Kiểm tra phương pháp ghép ma trận $[A|I]$ thông qua phép khử Gauss-Jordan.

In [50]:
def verify_inv(A, inv_nhom):
    if len(A) != len(A[0]):
        return "[ĐẠT]" if isinstance(inv_nhom, str) else "[CHƯA ĐẠT]"

    try:
        inv_np = np.linalg.inv(np.array(A, dtype=float))

        if isinstance(inv_nhom, str):
            return "[CHƯA ĐẠT] (Ma trận khả nghịch nhưng nhóm báo lỗi)"

        # Ép kiểu tường minh để tránh lỗi UFuncTypeError trên NumPy mới
        inv_nhom_np = np.array(inv_nhom, dtype=float)
        return "[ĐẠT]" if np.allclose(inv_nhom_np, inv_np, atol=1e-5) else "[CHƯA ĐẠT]"

    except np.linalg.LinAlgError:
        return "[ĐẠT]" if isinstance(inv_nhom, str) else "[CHƯA ĐẠT] (Ma trận suy biến)"
    except Exception:
        return "[CHƯA ĐẠT]"


def test_inverse():
    print("\n" + "="*60)
    print("--- TEST HÀM TÍNH NGHỊCH ĐẢO (inverse) ---")
    print("="*60)

    A1 = [[4, 7], [2, 6]]
    print("TC1 (2x2):")
    print(f"  + Kết quả : {inverse(A1)}")
    print(f"  + Đánh giá     : {verify_inv(A1, inverse(A1))}\n")

    A2 = [[1, 1, 1], [0, 2, 5], [2, 5, -1]]
    print("TC2 (3x3):")
    print(f"  + Kết quả : {inverse(A2)}")
    print(f"  + Đánh giá     : {verify_inv(A2, inverse(A2))}\n")

    A3 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
    print("TC3 (Suy biến không có nghịch đảo):")
    print(f"  + Kết quả : {inverse(A3)}")
    print(f"  + Đánh giá     : {verify_inv(A3, inverse(A3))}\n")

    A4 = [[1, 0], [0, 1]]
    print("TC4 (Ma trận đơn vị):")
    print(f"  + Kết quả : {inverse(A4)}")
    print(f"  + Đánh giá     : {verify_inv(A4, inverse(A4))}\n")

    A5 = [[1, 2], [3, 4], [5, 6]]
    print("TC5 (Ma trận không vuông):")
    print(f"  + Kết quả : {inverse(A5)}")
    print(f"  + Đánh giá     : {verify_inv(A5, inverse(A5))}\n")

### 2.4. Kiểm thử Trích xuất Hạng và Cơ sở
Đảm bảo thuật toán xác định chính xác số lượng vector độc lập tuyến tính.

In [51]:
def verify_rank(A, res_nhom):
    rank_np = np.linalg.matrix_rank(np.array(A, dtype=float))
    return "[ĐẠT]" if res_nhom['rank'] == rank_np else f"[CHƯA ĐẠT]"

def test_rank_and_basis():
    print("\n" + "="*60)
    print("--- TEST HÀM HẠNG & CƠ SỞ (rank_and_basis) ---")
    print("="*60)

    def print_res(res, tc_name):
        print(f"{tc_name}:")
        print(f"  + Hạng  : {res['rank']}")
        print(f"  + Cơ sở dòng   : {res['row_space_basis']}")
        print(f"  + Cơ sở cột    : {res['column_space_basis']}")

    A1 = [[1, 2], [3, 4]]
    res1 = rank_and_basis(A1)
    print_res(res1, "TC1 (Đầy đủ hạng)")
    print(f"  + Đánh giá     : {verify_rank(A1, res1)}\n")

    A2 = [[1, 2, 3], [2, 4, 6]]
    res2 = rank_and_basis(A2)
    print_res(res2, "TC2 (Khuyết hạng)")
    print(f"  + Đánh giá     : {verify_rank(A2, res2)}\n")

    A3 = [[0, 0], [0, 0]]
    res3 = rank_and_basis(A3)
    print_res(res3, "TC3 (Ma trận toàn số 0)")
    print(f"  + Đánh giá     : {verify_rank(A3, res3)}\n")

    A4 = [[1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 0]]
    res4 = rank_and_basis(A4)
    print_res(res4, "TC4 (Chữ nhật đứng 4x3)")
    print(f"  + Đánh giá     : {verify_rank(A4, res4)}\n")

    A5 = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
    res5 = rank_and_basis(A5)
    print_res(res5, "TC5 (Chữ nhật ngang 3x4)")
    print(f"  + Đánh giá     : {verify_rank(A5, res5)}\n")

### 2.5. Kiểm thử Giải ma trận tam giác trên


### TEST ALL

In [52]:
if __name__ == "__main__":
    test_gaussian_eliminate()
    test_determinant()
    test_inverse()
    test_rank_and_basis()
    test_back_substitution()
    print("="*60)


--- TEST HÀM GIẢI HỆ PHƯƠNG TRÌNH (gaussian_eliminate) ---
TC1 (Nghiệm duy nhất):
  + Kết quả : [2.0, 3.0, -0.9999999999999999]
  + Đánh giá     : [ĐẠT]

Khong co pivot tai cot 2
TC2 (Vô số nghiệm):
  + Kết quả : Vo so nghiem
  + Đánh giá     : [ĐẠT]

Khong co pivot tai cot 2
TC3 (Vô nghiệm):
  + Kết quả : Vo nghiem
  + Đánh giá     : [ĐẠT]

Khong co pivot tai cot 2
TC4 (Ma trận chữ nhật 4x3):
  + Kết quả : Vo nghiem
  + Đánh giá     : [ĐẠT]

Khong co pivot tai cot 0
TC5 (Suy biến cột đầu):
  + Kết quả : Vo so nghiem
  + Đánh giá     : [ĐẠT]


--- TEST HÀM TÍNH ĐỊNH THỨC (determinant) ---
TC1 (Ma trận 2x2)           | Det: -2.0   | [ĐẠT]
TC2 (Ma trận đường chéo)    | Det: 24.0   | [ĐẠT]
Khong co pivot tai cot 2
TC3 (Suy biến det=0)        | Det: 0.0    | [ĐẠT]
TC4 (Ma trận đơn vị 4x4)    | Det: 1.0    | [ĐẠT]
TC5 (Ma trận không vuông)   | Det: Ma tran khong vuong, khong tinh duoc dinh thuc | [ĐẠT]

--- TEST HÀM TÍNH NGHỊCH ĐẢO (inverse) ---
TC1 (2x2):
  + Kết quả : [[0.600000000000000